# Template Inversion Evaluation

This notebook evaluates the robustness of the proposed anonymization
algorithms against template inversion attacks.

A trained SciFive-based embedding inversion model is used to reconstruct
clinical text from anonymized embeddings.

The reconstruction quality is evaluated using:

- BLEU
- BERTScore-F1

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch

from src.inversion import (
    load_or_train_inversion_model,
    evaluate_embeddings,
)

d:\Post UG\M.Tech\FYP\Syntactic Vector Anonymization\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
DATA_DIR = PROJECT_ROOT / "data"

EMBEDDING_PATH = DATA_DIR / "case_embeddings.parquet"
TEXT_PATH = DATA_DIR / "case_texts.parquet"

ANON_DIR = DATA_DIR / "anonymized_outputs"

RESULTS_DIR = DATA_DIR / "results" / "template_inversion"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_DIR = PROJECT_ROOT / "saved_inversion_model"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
embeddings = pd.read_parquet(EMBEDDING_PATH).to_numpy(dtype=np.float32)
print(embeddings.shape)

texts_df = pd.read_parquet(TEXT_PATH)[:100]
print(texts_df.shape)

(100, 768)
(100, 3)


In [4]:
def truncate_to_core(
    text,
    max_sentences=5,
):
    sentences = [
        s.strip()
        for s in text.split(".")
        if len(s.strip()) > 20
    ]

    return ". ".join(
        sentences[:max_sentences]
    ) + "."


texts_df["case_text"] = (
    texts_df["case_text"]
    .fillna("")
)

texts_df["truncated_text"] = (
    texts_df["case_text"]
    .apply(truncate_to_core)
)

mapping_path = DATA_DIR / "text_mapping.csv"

texts_df[
    [
        "case_id",
        "case_text",
        "truncated_text",
    ]
].to_csv(
    mapping_path,
    index=False,
)

texts = texts_df[
    "truncated_text"
].tolist()

print(
    "Total texts:",
    len(texts),
)

print("\nExample:\n")
print(texts[0])

Total texts: 100

Example:

A 53-year-old woman presented with a 10-year history of intermittent abdominal pain, swelling and continuous vomiting. The patient denied presence of fever, nausea, and weight loss. There were no significant findings at physical examination. An abdominal ultrasound exam revealed a 10. 0 cm mass of heterogeneous echogenicity in the left upper abdomen.


In [ ]:
tokenizer, model, adapter, device = (
    load_or_train_inversion_model(
        embeddings=embeddings,
        texts=texts,
        model_dir=MODEL_DIR,
        epochs=50,
        batch_size=16,
        learning_rate=3e-4,
        latent_len=48,
        device=device,
    )
)


Found existing inversion model.


d:\Post UG\M.Tech\FYP\Syntactic Vector Anonymization\src\inversion\model.py:131: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(


In [7]:
summary = []

summary.append(
    evaluate_embeddings(
        embeddings=embeddings,
        reference_texts=texts,
        tokenizer=tokenizer,
        model=model,
        adapter=adapter,
        device=device,
        label="Baseline",
        output_dir=RESULTS_DIR / "baseline",
    )
)

d:\Post UG\M.Tech\FYP\Syntactic Vector Anonymization\src\inversion\metrics.py:71: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\torch\csrc\utils\tensor_numpy.cpp:212.)
  embeddings = torch.from_numpy(



Evaluating Baseline


Baseline: 100%|██████████| 100/100 [07:09<00:00,  4.30s/it]


      label      bleu  bertscore_f1
0  Baseline  2.475401      0.585424


In [8]:
CLUSTER_K = [5, 10, 25, 50, 100]

for k in CLUSTER_K:

    print(f"\nMDAV-C | k={k}")

    anon_embeddings = (
        pd.read_parquet(
            ANON_DIR /
            f"mdav_c_k{k}.parquet"
        )
        .to_numpy(dtype=np.float32)
    )

    summary.append(
        evaluate_embeddings(
            embeddings=anon_embeddings,
            reference_texts=texts,
            tokenizer=tokenizer,
            model=model,
            adapter=adapter,
            device=device,
            label=f"MDAV-C-k{k}",
            output_dir=(
                RESULTS_DIR /
                "mdav_c" /
                f"k_{k}"
            ),
        )
    )


MDAV-C | k=5

Evaluating MDAV-C-k5


MDAV-C-k5: 100%|██████████| 100/100 [07:45<00:00,  4.65s/it]


       label      bleu  bertscore_f1
0  MDAV-C-k5  2.568397      0.583106

MDAV-C | k=10

Evaluating MDAV-C-k10


MDAV-C-k10: 100%|██████████| 100/100 [07:49<00:00,  4.70s/it]


        label      bleu  bertscore_f1
0  MDAV-C-k10  2.904097      0.586482

MDAV-C | k=25

Evaluating MDAV-C-k25


...

        label      bleu  bertscore_f1
0  MDAV-C-k25  2.561052      0.583205

MDAV-C | k=50

Evaluating MDAV-C-k50


MDAV-C-k50: 100%|██████████| 100/100 [08:14<00:00,  4.95s/it]


        label      bleu  bertscore_f1
0  MDAV-C-k50  2.792898      0.585332

MDAV-C | k=100

Evaluating MDAV-C-k100


MDAV-C-k100: 100%|██████████| 100/100 [08:46<00:00,  5.27s/it]


         label      bleu  bertscore_f1
0  MDAV-C-k100  2.724003      0.587178


In [9]:
for k in CLUSTER_K:

    print(f"\nRMDAV-M | k={k}")

    anon_embeddings = (
        pd.read_parquet(
            ANON_DIR /
            f"rmdav_m_k{k}.parquet"
        )
        .to_numpy(dtype=np.float32)
    )

    summary.append(
        evaluate_embeddings(
            embeddings=anon_embeddings,
            reference_texts=texts,
            tokenizer=tokenizer,
            model=model,
            adapter=adapter,
            device=device,
            label=f"RMDAV-M-k{k}",
            output_dir=(
                RESULTS_DIR /
                "rmdav_m" /
                f"k_{k}"
            ),
        )
    )


RMDAV-M | k=5

Evaluating RMDAV-M-k5


RMDAV-M-k5: 100%|██████████| 100/100 [08:13<00:00,  4.94s/it]


        label      bleu  bertscore_f1
0  RMDAV-M-k5  2.284965      0.582957

RMDAV-M | k=10

Evaluating RMDAV-M-k10


RMDAV-M-k10: 100%|██████████| 100/100 [08:03<00:00,  4.84s/it]


         label      bleu  bertscore_f1
0  RMDAV-M-k10  2.271187       0.58107

RMDAV-M | k=25

Evaluating RMDAV-M-k25


...

         label      bleu  bertscore_f1
0  RMDAV-M-k25  2.362788      0.585133

RMDAV-M | k=50

Evaluating RMDAV-M-k50


RMDAV-M-k50: 100%|██████████| 100/100 [06:54<00:00,  4.14s/it]


         label      bleu  bertscore_f1
0  RMDAV-M-k50  2.367896      0.584236

RMDAV-M | k=100

Evaluating RMDAV-M-k100


RMDAV-M-k100: 100%|██████████| 100/100 [06:58<00:00,  4.19s/it]


          label      bleu  bertscore_f1
0  RMDAV-M-k100  2.360329      0.582242


In [10]:
summary_df = pd.DataFrame(summary)

summary_df.to_csv(
    RESULTS_DIR /
    "summary.csv",
    index=False,
)

summary_df

,label,bleu,bertscore_f1
0,Baseline,2.475401,0.585424
1,MDAV-C-k5,2.568397,0.583106
2,MDAV-C-k10,2.904097,0.586482
3,MDAV-C-k25,2.561052,0.583205
4,MDAV-C-k50,2.792898,0.585332
5,MDAV-C-k100,2.724003,0.587178
6,RMDAV-M-k5,2.284965,0.582957
7,RMDAV-M-k10,2.271187,0.581070
8,RMDAV-M-k25,2.362788,0.585133
9,RMDAV-M-k50,2.367896,0.584236
